In [4]:
__author__ = 'Cyrille Mvomo, https://github.com/cyrillemvomo'
__version__ = "1"
__license__ = "MIT"

**Import**

In [5]:
import sys
sys.path.append("/Users/cyrilleetude/Documents/GitHub/WhiteBoxDL/Preprocessing/1_Get_Data/MJFF_Cohort/1_Transform")
import numpy as np
import pandas as pd
import copy
import pickle
from scipy.signal import detrend

**Read**

In [6]:
with open("/Users/cyrilleetude/Documents/GitHub/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/2_Transform/1_Signal_Processing/Raw_csv_HomeDay2/EXTRACTED_DATA.bin", "rb") as f:
    EXTRACTED_DATA_HOME_RAW = pickle.load(f)

**Split into 5 second bouts**

In [7]:
# function to handle nan

def detrend_1d_array_with_nan(array):
    nan_indices = np.where(array * 0 !=0)[0]
    inf_indices = np.where(np.isinf(array))[0]
    nan_indices = np.unique(np.concatenate((nan_indices, inf_indices)))
    Data_no_nans = np.copy(array)
    Data_no_nans[nan_indices] = 0
    Data_no_nans = detrend(Data_no_nans)  # Detrend the data
    #Restore NaNs to their original positions
    Data_no_nans[nan_indices] = np.nan
    return Data_no_nans

In [8]:
EXTRACTED_DATA_SPLITTED = {}
sampling_rate = 50 #hz
count_bouts = 0
for participant in list(EXTRACTED_DATA_HOME_RAW.keys()):
    participant_data = []
    for sequence_id in range(len(EXTRACTED_DATA_HOME_RAW[participant])):
        df = EXTRACTED_DATA_HOME_RAW[participant][sequence_id].reset_index(drop=True)
        five_s_bouts_indexes = np.array_split(df.index.values, len(df)//(5*sampling_rate))
        for bout_id in range(len(five_s_bouts_indexes)):
            array = df.iloc[five_s_bouts_indexes[bout_id]].ACC_VERTICAL_MS2.values
            participant_data.append(detrend_1d_array_with_nan(array = array))
    EXTRACTED_DATA_SPLITTED[participant] = participant_data



In [9]:
# Save as bin file 
with open("/Users/cyrilleetude/Documents/GitHub/WhiteBoxDL/Preprocessing/1_Get_Data/MJFF_Cohort/1_Transform/temp/Extracted_Data_MJFF_HOME_2.bin", "wb") as f:
            pickle.dump(EXTRACTED_DATA_SPLITTED, f)